# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable XGBOOST/CATBOOST Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [ ]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'portable_m5_transfer_auto'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    available = sorted(p.name for p in REPORTS_DIR.glob(f'{preferred_tag}_*.csv'))
    raise FileNotFoundError(
        f'Missing transfer report: {preferred.name}. Run the transfer cell above first. '
        f'Available transfer CSVs for tag {preferred_tag!r}: {available}'
    )


## Run Transfer Evaluation

In [ ]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'P' / 'xgboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets P --models XGBOOST --tag "{TAG}" --calibration recent_level_auto


## Summary

In [ ]:
!python "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py" \
  --m5-dir "/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy" \
  --granularity dept_store \
  --datasets P \
  --models XGBOOST \
  --tag "portable_m5_transfer_auto" \
  --calibration recent_level_auto


In [35]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)


,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,0,10080,0.137533,3997.124824,-1474.620971,2.209112,1164.390318,0.723413


## Detailed Horizon Metrics

In [36]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,1,840,0.100423,3003.405348,-857.760097,1.526551,850.206950,0.654762
1,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,2,840,0.131516,4158.539834,-1643.706109,1.833990,1113.450713,0.690476
2,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,3,840,0.118487,3674.315222,-1254.092783,1.892350,1003.144666,0.682143
3,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,4,840,0.127756,3715.261170,-1531.171009,2.041622,1081.612847,0.709524
4,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,5,840,0.147459,4451.842254,-1979.959608,2.218219,1248.427317,0.735714
5,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,6,840,0.131662,3858.828576,-1207.948644,2.123364,1114.682936,0.680952
6,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,7,840,0.137405,3997.589554,-1286.076114,2.258776,1163.307883,0.686905
7,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,8,840,0.144837,4198.193938,-1573.987126,2.383661,1226.223788,0.745238
8,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,9,840,0.143473,4173.774753,-1615.532008,2.373662,1214.679870,0.757143
9,M5_TRANSFER_P,XGBOOST_recent_level_blend,test,10,840,0.145794,4033.101666,-1510.326082,2.482958,1234.333277,0.767857


## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts